# Construction du dataset DPO

Objectif : construire le dataset DPO final à partir d'UltraMedical-Preference (seule source disponible pour le DPO, voir `notebooks/01_exploration_sources.ipynb`).

Deux nettoyages s'appliquent, dans `load_ultramedical_preference` et `clean_dpo_dataset` (`scripts/extraction.py`) :

- la fuite train/validation déjà repérée dans le notebook d'exploration (1209 `prompt_id` partagés entre train et validation) : les lignes concernées sont retirées du train au chargement, décision documentée dans `docs/decisions.md`.
- les doublons exacts (prompt, chosen, rejected), pas encore explorés jusqu'ici : ce notebook les quantifie et documente leur retrait.

In [1]:
import sys
sys.path.append("..")

from collections import Counter, defaultdict
from dotenv import load_dotenv
from scripts.extraction import load_ultramedical_preference, clean_dpo_dataset, build_dpo_dataset

load_dotenv()

/home/rapha/ia-engineer/llm-finetuning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## Chargement, fuite train/validation déjà retirée

`load_ultramedical_preference` retire déjà du train les `prompt_id` partagés avec validation. On vérifie que la fuite est bien à zéro avant d'aller plus loin.

In [2]:
brut = load_ultramedical_preference()
print("lignes chargées (fuite déjà retirée) :", len(brut))

repartition = Counter(r["split"] for r in brut)
for split, n in repartition.items():
    print(f"{split:12s} {n:7d}")

lignes chargées (fuite déjà retirée) : 110753
train         107744
validation      2232
test             777


In [3]:
# la fuite documentée porte sur prompt_id (pas conservé dans les records du pipeline),
# on la revérifie ici directement sur le dataset source
from datasets import load_dataset
source = load_dataset("TsinghuaC3I/UltraMedical-Preference")
ids = {name: set(split_data["prompt_id"]) for name, split_data in source.items()}
print("fuite train/validation :", len(ids["train"] & ids["validation"]))

fuite train/validation : 1209


## Doublons exacts (prompt, chosen, rejected)

Pas encore vérifiés jusqu'ici (le notebook d'exploration ne portait que sur la fuite de `prompt_id`). Un même prompt a normalement plusieurs paires chosen/rejected différentes (c'est le principe du DPO), mais un triple identique répété plusieurs fois est un vrai doublon.

In [4]:
cles = [(r["prompt"], r["chosen"], r["rejected"]) for r in brut]
compteur = Counter(cles)
doublons = {cle: n for cle, n in compteur.items() if n > 1}

print("triples uniques en double :", len(doublons))
print("lignes en trop à retirer :", sum(n - 1 for n in doublons.values()))
print("prompts uniques :", len(set(r["prompt"] for r in brut)), "/", len(brut), "lignes")

triples uniques en double : 12285
lignes en trop à retirer : 12285
prompts uniques : 78270 / 110753 lignes


Un exemple de triple dupliqué, pour voir à quoi ça ressemble :

In [5]:
exemple = next(iter(doublons))
print("prompt  :", exemple[0][:200].replace("\n", " "), "...")
print("chosen  :", exemple[1][:150].replace("\n", " "), "...")
print("rejected:", exemple[2][:150].replace("\n", " "), "...")
print("répété", doublons[exemple], "fois")

prompt  : A lady comes with melanotic pigmentation of lip, presence of multiple polyps in the intestine, and a positive family history. What is the most probable diagnosis?  A. Peutz-Jegher's Syndrome B. Gardne ...
chosen  : Let's think step by step.  The patient has melanotic pigmentation of the lip, which is a characteristic feature of Peutz-Jeghers syndrome. Additionall ...
rejected: Let's break down the symptoms:  * Melanotic pigmentation of the lip: This is a common feature of several genetic syndromes. * Presence of multiple pol ...
répété 2 fois


Ce sont bien des lignes strictement identiques (même prompt, même réponse choisie, même réponse rejetée), pas des variantes utiles à garder pour le DPO : à dédoublonner en gardant la première occurrence, comme pour l'agrégat SFT dans `notebooks/02_construction_sft.ipynb`.

## Vérification : les doublons ne traversent pas les splits

Important à vérifier avant de dédoublonner : si un même triple apparaissait à la fois en train et en validation, le retirer uniquement du train (logique de `clean_dpo_dataset`, qui garde la première occurrence rencontrée) romprait l'indépendance de la validation, comme pour la fuite de `prompt_id`.

In [6]:
splits_par_triple = defaultdict(set)
for r in brut:
    cle = (r["prompt"], r["chosen"], r["rejected"])
    splits_par_triple[cle].add(r["split"])

cross_split = {cle: s for cle, s in splits_par_triple.items() if len(s) > 1}
print("triples dupliqués à cheval sur plusieurs splits :", len(cross_split))

triples dupliqués à cheval sur plusieurs splits : 0


Zéro triple à cheval sur plusieurs splits : tous les doublons sont internes à un seul split. Le dédoublonnage peut se faire sans risque de fuite.

## Dataset DPO final

`build_dpo_dataset` charge UltraMedical-Preference (fuite déjà retirée) puis applique `clean_dpo_dataset`, qui retire les doublons exacts (prompt, chosen, rejected) en gardant la première occurrence.

In [7]:
final = build_dpo_dataset()
print("brut (fuite retirée)     :", len(brut))
print("final (dédoublonné)      :", len(final))
print("doublons retirés         :", len(brut) - len(final))

repartition_finale = Counter(r["split"] for r in final)
for split, n in repartition_finale.items():
    print(f"{split:12s} {n:7d}")

brut (fuite retirée)     : 110753
final (dédoublonné)      : 98468
doublons retirés         : 12285
train          95464
validation      2228
test             776


## Synthèse

Dataset DPO construit à partir d'UltraMedical-Preference (seule source disponible, licence MIT, anglais, voir `docs/sources.md`) : fuite train/validation retirée (1609 lignes), puis 12285 triples (prompt, chosen, rejected) dupliqués dédoublonnés (24570 lignes concernées, 12285 lignes retirées, aucun de ces doublons ne traversant deux splits). Les splits train, validation et test hérités de la source sont conservés tels quels, pas besoin de les reconstruire comme pour le SFT (UltraMedical-Preference est une source unique, déjà correctement dimensionnée).

Prochaine étape : anonymiser SFT et DPO avec Presidio.